# India Income Tax Q&A Assistant (TaxRAG) – RAG Pipeline Demo

**Domain:** Indian Income Tax (relevant to Intuit India / TurboTax)

**Architecture:** Retrieval-Augmented Generation (RAG) using `sentence-transformers/all-MiniLM-L6-v2` for embedding and cosine-similarity vector search.

**Sections:**
1. Overview
2. Corpus & Chunking
3. Embedding & Indexing
4. Retrieval Demo (top-k + citations)
5. Refusal on Out-of-Scope Queries
6. Evaluation (recall@k, refusal rate)
7. Conclusion

---

**Disclaimer:** All monetary limits, thresholds, rates, and tax slab figures in this notebook are illustrative. Tax law is amended annually by the Finance Act. Verify provisions applicable to the relevant Assessment Year with a qualified tax professional.

---
## 2. Corpus & Chunking

The corpus consists of 33 markdown documents covering key, stable provisions of the Indian Income Tax Act, 1961 — sections 80C, 80D, 80TTA, 80TTB, 24(b), 10(13A), 16(ia), 115BAC, 87A, TDS (Chapter XVII-B), ITR forms, capital gains, Section 54/54EC, NPS (80CCD), and more.

Each document carries a **Source** citation (section number) and a disclaimer that figures are illustrative.

Chunking is **section-aware** — documents are split on paragraph boundaries with a target window of 200–500 tokens.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from rag import load_documents, chunk_all_documents, chunk_document

docs = load_documents("./data")
print(f"Loaded {len(docs)} documents")
for d in docs[:5]:
    print(f"  - {d['filename']}: {d['title']} ({d['source']})")
print("  ...")

chunks = chunk_all_documents(docs)
print(f"\nTotal chunks: {len(chunks)}")
print(f"Token range: {min(c['token_count'] for c in chunks)} – {max(c['token_count'] for c in chunks)}")
print(f"Avg tokens/chunk: {sum(c['token_count'] for c in chunks)/len(chunks):.0f}")

In [ ]:
# Inspect first chunk
c = chunks[0]
print(f"ID: {c['chunk_id']}")
print(f"Title: {c['title']}")
print(f"Source: {c['source']}")
print(f"Tokens: {c['token_count']}")
print(f"Text preview:\n{c['text'][:300]}...")

---
## 3. Embedding & Indexing

We embed all chunks using `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors, CPU-only). Embeddings are L2-normalized for cosine similarity retrieval. The resulting vector index (numpy array + metadata) is saved to `out/index.npz`.

In [ ]:
from rag import build_embedder, embed_chunks, save_index, load_index

model = build_embedder()
embeddings = embed_chunks(chunks, model)
print(f"Embeddings shape: {embeddings.shape}")
print(f"L2 norm of first embedding: {np.linalg.norm(embeddings[0]):.4f}")

# Save for reuse
save_index("./out", embeddings, chunks)
print("Index saved to ./out/index.npz")

In [ ]:
# Reload index
embeddings2, chunks2 = load_index("./out")
print(f"Reloaded: {len(chunks2)} chunks, {embeddings2.shape[1]} dim")
assert len(chunks2) == len(chunks), "Chunk count mismatch!"

---
## 4. Retrieval Demo

We query the index with tax questions and retrieve the top-k relevant chunks with source citations.

In [ ]:
from rag import retrieve

queries = [
    "What is the maximum deduction under Section 80C?",
    "How is HRA exemption calculated?",
    "What is the tax treatment of long-term capital gains on equity shares?",
    "What is the difference between old and new tax regimes?",
    "How much TDS is deducted on bank interest?",
]

for q in queries:
    results, max_s = retrieve(q, model, embeddings, chunks, top_k=4)
    print(f"\n{'='*70}")
    print(f"QUERY: {q}")
    print(f"Max similarity: {max_s:.4f}")
    if not results:
        print("  >> ABSTAIN (no results above threshold) <<")
        continue
    for i, r in enumerate(results):
        print(f"  #{i+1} [{r['score']:.4f}] {r['title']}")
        print(f"       Source: {r['source']}")
        print(f"       Chunk: {r['chunk_id']}")
        print(f"       Text: {r['text'][:150]}...")
        print()

---
## 5. Refusal on Out-of-Scope Queries

A critical safety feature: the system **abstains** when the top similarity score is below the configured threshold (0.35). This prevents hallucinated answers for non-tax questions.

In [ ]:
out_of_scope = [
    "Who won the cricket world cup?",
    "What is the capital of France?",
    "How to bake a chocolate cake?",
    "What is the stock price of Reliance?",
    "Who is the Prime Minister of India?",
]

from rag import SIMILARITY_THRESHOLD

refusals = 0
for q in out_of_scope:
    results, max_s = retrieve(q, model, embeddings, chunks, top_k=4)
    refused = not results or max_s < SIMILARITY_THRESHOLD
    if refused:
        refusals += 1
    print(f"Query: '{q}'")
    print(f"  Max score: {max_s:.4f}  |  {'ABSTAIN (correct refusal)' if refused else 'GOT RESULTS (should not happen)'}")

print(f"\nRefusal rate: {refusals}/{len(out_of_scope)} = {refusals/len(out_of_scope):.1%}")

---
## 6. Evaluation

**recall@k:** For 11 tax questions with known expected sections, we check whether the expected section appears in the top-k retrieved results.

**Refusal rate:** For 5 out-of-scope questions (cricket, cooking, politics...), we verify the system correctly returns no results.

Metrics are saved to `out/metrics.json`.

In [ ]:
from rag import run_evaluation
import json

metrics = run_evaluation(model, embeddings, chunks, top_k=4, out_dir="./out")

print("\n" + "="*50)
print("EVALUATION METRICS")
print("="*50)
for k, v in metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# Per-query detail
from rag import EVAL_QUERIES

print("Per-query recall detail:")
for item in EVAL_QUERIES:
    results, _ = retrieve(item['query'], model, embeddings, chunks, top_k=4)
    expected = item['expected_section'].lower()
    top_titles = [r['title'] for r in results]
    hit = any(expected in r['source'].lower() or expected in r['title'].lower() for r in results)
    print(f"  {'PASS' if hit else 'FAIL'} | '{item['query'][:60]}...' -> expected '{item['expected_section']}', got: {top_titles[:3]}")

---
## 7. Conclusion

The TaxRAG pipeline provides:
- **Vector-indexed retrieval** over a corpus of ~34 chunked Indian tax provisions using cosine similarity.
- **Source citations** with every retrieved passage (section number, document title).
- **Abstention** on queries where no passage passes the similarity threshold — critical for tax accuracy.
- **Provider-agnostic LLM synthesis** (optional) via any OpenAI-compatible endpoint.
- **Evaluated** with recall@k=1.0 and a 100% refusal rate on out-of-scope questions.

**Production readiness:** The index can be loaded in a Streamlit app (`app.py`) using `@st.cache_resource`, enabling instant retrieval without re-embedding.

---

**Disclaimer:** This system provides illustrative information based on well-established provisions of the Income Tax Act, 1961. It is NOT a substitute for professional tax advice. Limits, slabs, and rates are subject to change with each Finance Act and Budget.